In [0]:
#Compare Source and Target Row Counts with zip-Given ordered lists of source table names, source row counts, and target row counts, generate reconciliation records indicating whether each table passed or failed count validation. Handle missing entries in any input collection.

#imports
import re
from typing import Final, Any, Literal
from dataclasses import dataclass, asdict
from itertools import zip_longest
from collections.abc import Iterable

#global variables
TABLE_NAME_PATTERN:Final[re.Pattern[str]]=re.compile(r"[A-Za-z][A-Za-z0-9_]*")
MISSING:Final=object()
#dataclass to track errors
@dataclass(frozen=True, slots=True)
class ReconciliationRecord:
    index:int
    table_name:str|None
    source_count:int|None
    target_count:int|None
    status:Literal["Pass","Fail"]
    reason:str
#iterable validation
def validate_collection(values:Any, name:str)->Iterable[object]:
    if isinstance(values, (str, bytes, bytearray)) or not isinstance(values, Iterable):
        raise TypeError(
            f"{name} must be an iterable such as list or tuple"
            f"received type {type(values).__name__}")
    return values 
#tabel validation
def validate_table_name(value:object, index:int)->tuple[str|None, str|None]:
    if value is MISSING:
        return None, f"missing source table name at index {index}"
    if not isinstance(value, str):
        return None, (
            f"table name at index {index} must be string"
            f"received {type(value).__name__}"
        )
    table_name=value.strip().lower()
    if not table_name:
        return None, f"table name at index {index} cannot be empty" 
    if not TABLE_NAME_PATTERN.fullmatch(table_name):
        return None, f"invalid table name at index {index}: {value!r}"
    return table_name, None
#count validation
def validate_count(value:object, count_type:str, index:int)->tuple[int|None, str|None]:
    if value is MISSING:
        return None, f"missing {count_type} count at index {index}"
    if isinstance(value, bool) or not isinstance(value, int):
        return None,(
            f"{count_type} count at index {index} must be an integer"
            f"received {type(value).__name__}"
        )
    if value<0:
        return None, f"{count_type} count at index {index} cannot be negative {value}"

    return value, None
#compare results
def compare_results(
    source_tables:Iterable[object],
    source_counts:Iterable[object],
    target_counts:Iterable[object]
)->list[ReconciliationRecord]:
    source_tables=validate_collection(source_tables,"source_tables")
    source_counts=validate_collection(source_counts, "source_counts")
    target_counts=validate_collection(target_counts, "target_counts")

    results:list[ReconciliationRecord]=[]
    seen_tables:set[str]=set()

    for index, (raw_table, raw_source_count, raw_target_count) in enumerate(
        zip_longest(
            source_tables,
            source_counts,
            target_counts,
            fillvalue=MISSING
        )
    ):
        table_name, table_error=validate_table_name(raw_table, index)
        source_count, source_error=validate_count(
            raw_source_count, "source", index
        )
        target_count, target_error=validate_count(
            raw_target_count, "target", index
        )
        errors=[
            error for error in (table_error, source_error, target_error)
            if error is not None
        ]
        if table_name is not None:
            if table_name in seen_tables:
                errors.append(f"duplicate table name :{table_name}")
            else:
                seen_tables.add(table_name)
        if errors:
            status:Literal["PASS", "FAIL"]="FAIL"
            reason="; ".join(errors)
        elif source_count==target_count:
            status="PASS"
            reason="source and target count matches"
        else:
            status="FAIL"
            reason=(
                f"row-count mismatch: source={source_count}"
                f"target={target_count}"
            )
        results.append(
            ReconciliationRecord(
                index=index,
                table_name=table_name,
                source_count=source_count,
                target_count=target_count,
                status=status,
                reason=reason,
            )
        ) 
    return results
#main function
if __name__=="__main__":
    source_tables = ["customers", "orders"]
    source_counts = [100, 250]
    target_counts = [100, 250]
    reconciliation_results = compare_results(
        source_tables,
        source_counts,
        target_counts,
    ) 
    for record in reconciliation_results:
        print(asdict(record))


{'index': 0, 'table_name': 'customers', 'source_count': 100, 'target_count': 100, 'status': 'PASS', 'reason': 'source and target count matches'}
{'index': 1, 'table_name': 'orders', 'source_count': 250, 'target_count': 250, 'status': 'PASS', 'reason': 'source and target count matches'}


In [0]:
#Create a Date Parsing Utility — Create a utility module containing a reusable function that converts multiple incoming date formats into a standardized date value, then import and use it from a separate ingestion script.
#import
from datetime import datetime, date
from typing import Any
#date validation & normaliation
ACCEPTED_DATE_FORMATS=(
    "%Y-%m-%d", # 2026-09-06
    "%d-%m-%Y", # 06/09/2026
    "%m-%d-%Y",        # 09-06-2026
    "%B %d, %Y",       # September 06, 2026
    "%Y-%m-%dT%H:%M:%S"# 2026-09-06T06:13:00 (ISO)
)
def normalize_date(date_in:str)->date:
    #expecting input format as string
    if not isinstance(date_in, str):
        raise TypeError(
            f"incoming date expected as string found {type(date_in).__name__}")
    cleaned_value=date_in.strip().strip("'\"")
    if not cleaned_value:
        raise ValueError(
                f"date is invalid value" )
    #checking if its date or not
    for date_format in ACCEPTED_DATE_FORMATS:
        try:
            #return datetime.strptime(cleaned_value,date_format ).strftime('%Y-%m-%d')    
            return datetime.strptime(cleaned_value, date_format).date()        
        except ValueError:
            continue
    
#ingestion script_01
def data_ingestion(row:object)->str:
    if not isinstance(row,str):
        raise TypeError(f"expected row as string received {type(row).__name__}")
    
    column_split=row.split(",") 
    #we could validate separator, each string column value, amount as well 
    #row expected to be- "Ram, Ganesh, '04/03/2027', 345"
    birth_date=column_split[2]
    valid_birth_date = normalize_date(birth_date) 
    return row

if __name__=="__main__":
    rows=[
        "Ram, Ganesh, '04/03/2027', 345",
        "sam, sundar, '12/31/2027', 345",
        "raj, josh, '02/31/2027', 345"
    ]
for row in rows:
    result=data_ingestion(row)
    print(result)

Ram, Ganesh, '04/03/2027', 345
sam, sundar, '12/31/2027', 345
raj, josh, '02/31/2027', 345


In [0]:
#Create a Date Parsing Utility — Create a utility module containing a reusable function that converts multiple incoming date formats into a standardized date value, then import and use it from a separate ingestion script.

#date validation
from typing import Final
from datetime import date, datetime
ACCEPTED_DATE_FORMAT:Final[tuple[str,...]]=(
    '%m/%d/%Y',
    '%B %d, %Y', 
    '%Y-%m-%d',
    '%m-%d-%Y',
    "%Y-%m-%dT%H:%M:%S"
) 
class DateNormalizationError(ValueError):
    '''inherits all the properties of ValueError Class'''

def validate_date(in_date:object)->date:
    if not isinstance(in_date, str):
        raise TypeError(
            f"Incoming date must be a string;"
            f"received {type(in_date).__name__}"
        )
    cleaned_in_date=in_date.strip().strip("'\"")
    if not cleaned_in_date:
        raise DateNormalizationError(
            f"Incoming date cannot be empty." )
    for date_format in ACCEPTED_DATE_FORMAT:
        try:
            return datetime.strptime(in_date,ACCEPTED_DATE_FORMAT).date()
        except ValueError:
            continue
    accepted_formats=", ".join(ACCEPTED_DATE_FORMAT)
    raise DateNormalizationError(
        f"Invalid date found{cleaned_in_date!r}"
        f"Accepted formats: {accepted_formats}"
    ) 

In [0]:
'''%run '/Users/przirape7@gmail.com/Real Python-CGPT PGM/04.Conditions and Loops_Week36_2026/Improved_01_Normalization'''

from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
import csv
import logging
#from 04.Conditions and Loops_Week36_2026 import DateNormalizationError,validate_date

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
logger=logging.getLogger(__name__)
@dataclass(frozen=True)
class Records:
    first_name:str
    last_name:str
    birth_date:str
    amount:Decimal

def ingest_data(row:object)->Records:
    if not isinstance(row,str):
        raise TypeError(
            f"row expected to be string received {type(row).__name__}")
    cleaned_row=row.strip()
    if not cleaned_row:
        raise ValueError(f"row cannot be emplty")
    '''for csv.reader(row[record]):
        first_name, last_name, birth_date, amount=row[record]'''
    try:
        columns=next(csv.reader([row], skipinitialspace=True, strict=True))
    except csv.Error as error:
        raise ValueError(f"invalid csv row {row!r}")
    
    if len(columns)!=4:
        raise ValueError(
            f"Expected exactly 4 columns, but received {len(columns)}: {row!r}"
        )

    cleaned_first_name, cleaned_last_name, raw_birth_date, raw_amount=(
        column.strip() for column in columns
    )
    
    '''#validate first & last name
    if not isinstance(first_name, str) or not isinstance(last_name, str):
        raise TypeError(f"name is expected to be string received {type(first_name).__name__}"
                        f"and {type(last_name).__name__}")'''
        
    #cleaned_first_name=first_name.strip()
    #cleaned_last_name=last_name.strip()
    if not cleaned_first_name or not cleaned_last_name:
        raise ValueError(f"first name or last name cannot be empty")

    #validate birth date
    cleaned_date= validate_date(raw_birth_date)

    #validate amount
    '''if not isinstance(amunt, decimal):
        raise TypeError(f"amount expected to be decimal received {type(amount).__name__}")
    cleaned_amount=amount.strip()
    if not cleaned_amount:
        raise ValueError(f"amount cannot be empty")
    if cleaned_amount<0 :
        raise ValueError(f"amount cannot be negative")'''
        
    try:
        cleaned_amount=Decimal(raw_amount)
    except InvalidOperation as error:
        raise ValueError(
            f"amount must be numeric, received {raw_amount!r}") from error    
    #return record
    return Records(
        first_name=cleaned_first_name, 
        last_name=cleaned_last_name,
        birth_date=cleaned_date,
        amount=cleaned_amount
    )
    #test function data
    if __name__=="__main__":
        rows=[
            "Ram, Ganesh, '04/03/2027', 345",
            "Sam, Sundar, '12/31/2027', 345",
            "Raj, Josh, '02/31/2027', 345",
        ]
        for row_number, row in enumerate(rows, start=1):
            try:
                record=ingest_data(row)
                logger.info("successfully ingested row %s, %s,", row_number, record)
            except (TypeError, ValueError,DateNormalizationError ) as error:
                logger.error("rejected rows %s, %s", row_number, error)